# <u>Trending spot symmetry</u>


# 1) Importing modules

In [81]:
import numpy as np
# import pypyodbc
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns
from matplotlib.colors import to_hex

# 2) Data import + process

In [82]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx" # dummy datapath

df = pd.read_excel(data_path) # reading xlsx as df

basic_info = ["ADate", "MachineName", "Energy", "Device", "Gantry Angle", "Spot"] # basic spot info
grad_ratio = ["hor_rt_gradient", "hor_lt_gradient", "vert_rt_gradient", "vert_lt_gradient", "bltr_rt_gradient", "bltr_lt_gradient", "tlbr_rt_gradient", "tlbr_lt_gradient"] # gradient info of the spots

sub_df = df[basic_info + grad_ratio].copy() # new filtered df only with gradient data

## Setup - Calculate gradient ratio (GR) for each profile

In [83]:
# pixel coordinates of the expected spot position
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

# assigning pixel coordinates info
sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

# gradient ratio (-1 is when the spot is perfectly symmetrical)
sub_df['gr_hor'] = sub_df["hor_rt_gradient"] / sub_df["hor_lt_gradient"] # horizontal gradient ratio
sub_df['gr_vert']  = sub_df["vert_rt_gradient"] / sub_df["vert_lt_gradient"] # vertical gradient ratio
sub_df['gr_bltr']  = sub_df["bltr_rt_gradient"] / sub_df["bltr_lt_gradient"] # bottom-left to top-right gradient ratio
sub_df['gr_tlbr']  = sub_df["tlbr_rt_gradient"] / sub_df["tlbr_lt_gradient"] # top-left to bottom-right gradient ratio

## Setup - mean GR across all positions of the same energy

In [84]:
# mean df
avg_df = (
    sub_df
    .groupby(["ADate","Device", "MachineName", "Gantry Angle", "Energy"], as_index=False) # as_index makes sure there are entries in all info column
    .agg( # calculating means of all gradient ratio
        mean_gr_hor=("gr_hor", "mean"),
        mean_gr_vert=("gr_vert", "mean"),
        mean_gr_bltr=("gr_bltr", "mean"),
        mean_gr_tlbr=("gr_tlbr", "mean")
         )
)

# checking df
for column, content in avg_df.items():
    print(column)
#    for info, data in content.items():
#        print(info)

ADate
Device
MachineName
Gantry Angle
Energy
mean_gr_hor
mean_gr_vert
mean_gr_bltr
mean_gr_tlbr


Testing the avg_df works with selected_df in Savanna's graphing code

In [85]:
selected_df = avg_df[(avg_df["MachineName"]=="Gantry 2") & (avg_df["Device"] == "XRV-3000") & (avg_df['ADate'] >= "2025-01-01") & (avg_df["Energy"] == 70) &(avg_df["Gantry Angle"] == 180)] # arbitrary selections

#selected_df # checking df

# 3) Graphs

## Test 1.0 - Ploting the mean GR of all profiles in one gantry

- Starting from Savanna's base code
- Adding a time filter starting on YYYY/MM/DD and going back N months
- Changing the y intervals depends on the data range; 0.01 if it is smaller than 0.1, and 0.02 if it is larger than 0.1

In [86]:
def plotly_mean_gr(df, grad, gantry, device, energy, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module:
    - One energy at a particular gantry at a specific GA
    - Filter for XRV-3000 or XRV-4000
    - For each profile, the mean gradient ratio across all positions


    e.g. plotly_mean_gr(sub_df, "mean_gr_hor", "Gantry 1", "XRV-4000", 70, 0, "2026-01-01", 24)

    Input:
        df:             Pandas dataframe

        grad:           Column name of the dataframe in string (e.g. "gr_hor")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=grad,
        color_discrete_sequence= sns.color_palette("hls", 4).as_hex(),
        title=f'{gantry} - Mean Gradient Ratio across all spot positions for {energy} MeV',
        width=800,
        height=500
    )
    
    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=4,                 # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                    # thinner connecting lines
                    ))

    # Y-axis limit
    y_min = np.floor(selected_df[grad].min().min() * 100) / 100 # rounding minimum number to the lowest 0.01
    y_max = np.ceil(selected_df[grad].max().max() * 100) / 100 # rounding maximum number to the highest 0.01

    if y_max - y_min > 0.1:
        steps = 0.02 # y intervals will be 0.02 if the range of the data is over 0.1
    else:
        steps = 0.01 # y intervals will be 0.01 if the range of the data is smaller than 0.1

    fig.update_yaxes(range=[y_min,y_max], dtick=steps)

    # Show plot
    fig.show()

    return

In [87]:
# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 100 MeV spot, Gantry angle = 0, in last 2 months

mean_prof = ["mean_gr_hor", "mean_gr_vert", "mean_gr_bltr", "mean_gr_tlbr"]

plotly_mean_gr(avg_df, mean_prof, "Gantry 3", "XRV-3000", 100, 0, "2026-01-01", 24)

## Test 1.1 - Ploting the GR of one energy of all positions in one gantry at a particular gantry angle

- Lists of spot positions 

In [88]:
xrv3000_spot = ["Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Centre", "Bottom-Right"]

xrv4000_spot = ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right",
                "Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Centre", "Bottom-Right", 
                "Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]

postism_spot = ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right",
                "Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Right", 
                "Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]

- Making subplots for each position with all profiles

In [89]:
def plotly_all_prof(df, grad, gantry, device, energy, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module:
    - One energy at a particular gantry at a specific GA
    - Filter for XRV-3000 or XRV-4000
    - For each profile, the gradient ratio of each profile in all positions


    e.g. plotly_all_prof(sub_df, "gr_hor", "Gantry 1", "XRV-4000", 70, 0, "2026-01-01", 24)

    Input:
        df:             Pandas dataframe

        grad:           Column name of the dataframe in string; the different profiles (e.g. "gr_hor")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                     (df["Device"] == device) & 
                     (df['ADate'] >= start_date) & 
                     (df["Energy"] == energy) & 
                     (df["Gantry Angle"] == gantry_angle)
    ]

    # if data is from the XRV3000
    if device == "XRV-3000":

        # plot details
        subplot_titles = [f"{pos}" for pos in xrv3000_spot] # subplot titles

        clr_no = len(grad)
        colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours of the plots

        # defining the subplots inside the main plot
        fig = make_subplots(
            rows=3,
            cols=3,
            subplot_titles=subplot_titles,
            horizontal_spacing=0.05,
            vertical_spacing=0.075
        )

        # making subplot per position 
        for i, spot in enumerate(xrv3000_spot):

            # looping for row/col index
            total_cols = 3
            row = i // total_cols + 1
            col = i % total_cols + 1

            plot_df = selected_df[(selected_df["Spot"]==spot)] # plotting all 4 profiles in the same subplot

            # plot differnt profiles in each position 
            for profile, colour in zip(grad, colours):

                fig.add_trace(
                    go.Scatter(
                        x=plot_df['ADate'], 
                        y=plot_df[profile], 
                        name=profile, # labelling the lines
                        legendgroup=profile, # grouping the same profile into 1 legend
                        showlegend=(i==0), # ensure only 1 set of legend is shown
                        line=dict(color=colour) # colour details
                    ), 
                        row=row, 
                        col=col
                )
    
        # plot settings
        fig.update_layout(
            title=dict(
                text=f"<u>{gantry} - Gradient Ratio of {energy} MeV spots at GA{gantry_angle} across all positions on the {device}</u>",
                x=0.5
            ),
            width=1280,
            height=720,
            margin=dict(l=50, r=50, t=100, b=50),
        )

        # Optional: connect points by spot for clarity
        fig.update_traces(mode='markers+lines',
                        marker=dict(size=4, # marker size
                                    line=dict(width=2)), # outline width
                        line=dict(width=1) # thinner connecting lines
        )

        fig.show()

    # if data is for Post-ISM
    elif device == "XRV-4000":
            
        # plot details
        subplot_titles = [f"{pos}" for pos in xrv4000_spot] # subplot titles

        clr_no = len(grad)
        colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours of the plots

        # defining the subplots inside the main plot
        fig = make_subplots(rows=5,
                            cols=3,
                            subplot_titles=subplot_titles,
                            horizontal_spacing=0.05,
                            vertical_spacing=0.045
        )

        # making subplot per position 
        for i, spot in enumerate(xrv4000_spot):

            # looping for row/col index
            total_cols = 3
            row = i // total_cols + 1
            col = i % total_cols + 1

            plot_df = selected_df[(selected_df["Spot"]==spot)] # plotting all 4 profiles in the same subplot

            # plot differnt profiles in each position 
            for profile, colour in zip(grad, colours):

                fig.add_trace(
                    go.Scatter(
                        x=plot_df['ADate'], 
                        y=plot_df[profile], 
                        name=profile, # labelling the lines
                        legendgroup=profile, # grouping the same profile into 1 legend
                        showlegend=(i==0), # ensure only 1 set of legend is shown
                        line=dict(color=colour) # colour details
                    ), 
                        row=row, 
                        col=col
                )
        
        # plot settings
        fig.update_layout(
            title=dict(
                text=f"<u>{gantry} - Gradient Ratio of {energy} MeV spots at GA{gantry_angle} across all positions on the {device}</u>",
                x=0.5
            ),
            width=1280,
            height=1200,
            margin=dict(l=50, r=50, t=100, b=50),
        )

        # Optional: connect points by spot for clarity
        fig.update_traces(mode='markers+lines',
                        marker=dict(size=4, # marker size
                                    line=dict(width=2)), # outline width
                        line=dict(width=1) # thinner connecting lines
        )

        fig.show()

    return

- Testing for XRV3000

In [90]:
prof_label = ["gr_hor", "gr_vert", "gr_bltr", "gr_tlbr"]

plotly_all_prof(sub_df, prof_label, "Gantry 2", "XRV-3000", 100, 0, "2026-01-01", 24)

- Testing for XRV4000 specifally for Post-ISM

In [91]:
plotly_all_prof(sub_df, prof_label, "Gantry 1", "XRV-4000", 70, 0, "2026-01-01", 24)

## Test 1.2 - Ploting one energy in all positions for each GR in one gantry at a particular gantry angle

- List of profiles

In [92]:
profiles = ["gr_hor", "gr_vert", "gr_bltr", "gr_tlbr"]
prof_titles = ["Horizontal", "Vertical", "bltr", "tlbr"]

In [93]:
def plotly_all_pos(df, pos, gantry, device, energy, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module:
    - One energy at a particular gantry at a specific GA
    - Filter for XRV-3000 or XRV-4000
    - For each position, the gradient ratio in all profiles  


    e.g. plotly_all_pos(sub_df, "Top-Left", "Gantry 1", "XRV-4000", 70, 0, "2026-01-01", 24)

    Input:
        df:             Pandas dataframe

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                     (df["Device"] == device) & 
                     (df['ADate'] >= start_date) & 
                     (df["Energy"] == energy) & 
                     (df["Gantry Angle"] == gantry_angle)
    ]

    # plot details
    subplot_titles = [prof for prof in prof_titles] # subplot titles

    clr_no = len(pos)
    colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours of the plots

    # defining the subplots inside the main plot
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.05,
        vertical_spacing=0.1
    )

    # making subplot per position 
    for i, prof in enumerate(profiles):

        # looping for row/col index
        total_cols = 2
        row = i // total_cols + 1
        col = i % total_cols + 1

        for j, position in enumerate(pos):

            plot_df = selected_df[(selected_df["Spot"]==position)] # plotting all 9 spot positions in the same subplot

            fig.add_trace(
            go.Scatter(
                x=plot_df['ADate'], 
                y=plot_df[prof], 
                name=position, # labelling the lines
                legendgroup=position, # grouping the same profile into 1 legend
                showlegend=(i==0), # ensure only 1 set of legend is shown
                line=dict(color=colours[j]) # defining the colour of the plots according to the colour dict
            ), 
                row=row, 
                col=col
        )
            
    # plot settings
    fig.update_layout(
        title=dict(
            text=f"<u>{gantry} - Gradient Ratio of {energy} MeV spots at GA{gantry_angle} across all positions on the {device}</u>",
            x=0.5
        ),
        width=1280,
        height=720,
        margin=dict(l=50, r=50, t=100, b=50),
    )

    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=4, # marker size
                                line=dict(width=2)), # outline width
                    line=dict(width=1) # thinner connecting lines
    )

    fig.show()

    return

- Testing for XRV3000

In [94]:
xrv3000 = ["Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Centre", "Bottom-Right"]

plotly_all_pos(sub_df, xrv3000, "Gantry 2", "XRV-3000", 100, 0, "2026-01-01", 24)

- Testing for XRV4000

In [95]:
postism = ["Top-Top-Left", "Top-Top-Centre", "Top-Top-Right",
                "Top-Left", "Top-Centre", "Top-Right", 
                "Left", "Centre", "Right", 
                "Bottom-Left", "Bottom-Right", 
                "Bottom-Bottom-Left", "Bottom-Bottom-Centre", "Bottom-Bottom-Right"]

plotly_all_pos(sub_df, postism, "Gantry 1", "XRV-4000", 150, 0, "2026-01-01", 12)

## Test 1.3 - Ploting all energies of one GR in all positions in one gantry at a particular gantry angle 

In [96]:
def plotly_all_eng(df, grad, gantry, device, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module:
    - All energy at a particular gantry at a specific GA for one profile
    - Filter for XRV-3000 or XRV-4000
    - For each energy, the gradient ratio in all positions of one profile


    e.g. plotly_all_eng(sub_df, "gr_hor", "Gantry 1", "XRV-4000", 0, "2026-01-01", 24)

    Input:
        df:             Pandas dataframe

        grad:           Column name of the dataframe in string; the different profiles (e.g. "gr_hor")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                     (df["Device"] == device) & 
                     (df['ADate'] >= start_date) & 
                     (df["Gantry Angle"] == gantry_angle)
    ]

    # if data is from the XRV3000
    if device == "XRV-3000":

        # plot details
        subplot_titles = [f"{pos}" for pos in xrv3000_spot] # subplot titles

        # defining the subplots inside the main plot
        fig = make_subplots(
            rows=3,
            cols=3,
            subplot_titles=subplot_titles,
            horizontal_spacing=0.05,
            vertical_spacing=0.075
        )

        # making subplot per position 
        for i, spot in enumerate(xrv3000_spot):

            # looping for row/col index
            total_cols = 3
            row = i // total_cols + 1
            col = i % total_cols + 1

            plot_df = selected_df[(selected_df["Spot"]==spot)] # going through each spot positions

            energy_list = plot_df["Energy"].unique() # all the unique energies 
            clr_no = len(energy_list) # no of energies
            colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours of the plots

            for j, energy in enumerate(energy_list): # looping through all energies 

                prof_rad = plot_df[(plot_df["Energy"]==energy)] # the specify profile of 1 energy

                fig.add_trace(
                go.Scatter(
                    x=prof_rad['ADate'], 
                    y=prof_rad[grad], 
                    name=f"{energy} MeV", # labelling the lines
                    legendgroup=str(energy), # grouping the same profile into 1 legend
                    showlegend=(i==0), # ensure only 1 set of legend is shown
                    line=dict(color=colours[j]) # defining the colour of the plots according to the colour dict
                ), 
                    row=row, 
                    col=col
            )
        
        # plot settings
        grad_title = prof_titles[profiles.index(grad)] # gradient profile title from earlier list (Test 1.2)

        fig.update_layout(
            title=dict(
                text=f"<u>{gantry} - Gradient ratio ({grad_title}) of all energy spots at GA{gantry_angle} across all positions on the {device}</u>",
                x=0.5
            ),
            width=1280,
            height=720,
            margin=dict(l=50, r=50, t=100, b=50),
        )

        # Optional: connect points by spot for clarity
        fig.update_traces(mode='markers+lines',
                        marker=dict(size=4, # marker size
                                    line=dict(width=2)), # outline width
                        line=dict(width=1) # thinner connecting lines
        )
            
    # if data is for Post-ISM
    elif device == "XRV-4000":
            
        # plot details
        subplot_titles = [f"{pos}" for pos in xrv4000_spot] # subplot titles

        # defining the subplots inside the main plot
        fig = make_subplots(rows=5,
                            cols=3,
                            subplot_titles=subplot_titles,
                            horizontal_spacing=0.05,
                            vertical_spacing=0.045
        )

        # making subplot per position 
        for i, spot in enumerate(xrv4000_spot):

            # looping for row/col index
            total_cols = 3
            row = i // total_cols + 1
            col = i % total_cols + 1

            plot_df = selected_df[(selected_df["Spot"]==spot)] # going through each spot positions

            energy_list = plot_df["Energy"].unique() # all the unique energies 
            clr_no = len(energy_list) # no of energies
            colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours of the plots

            for j, energy in enumerate(energy_list): # looping through all energies 

                prof_rad = plot_df[(plot_df["Energy"]==energy)] # the specify profile of 1 energy

                fig.add_trace(
                go.Scatter(
                    x=prof_rad['ADate'], 
                    y=prof_rad[grad], 
                    name=f"{energy} MeV", # labelling the lines
                    legendgroup=str(energy), # grouping the same profile into 1 legend
                    showlegend=(i==0), # ensure only 1 set of legend is shown
                    line=dict(color=colours[j]) # defining the colour of the plots according to the colour dict
                ), 
                    row=row, 
                    col=col
            )
                    
        # plot settings
        grad_title = prof_titles[profiles.index(grad)] # gradient profile title from earlier list (Test 1.2)

        fig.update_layout(
            title=dict(
                text=f"<u>{gantry} - Gradient ratio ({grad_title}) of all energy spots at GA{gantry_angle} across all positions on the {device}</u>",
                x=0.5
            ),
            width=1280,
            height=1200,
            margin=dict(l=50, r=50, t=100, b=50),
        )

        # Optional: connect points by spot for clarity
        fig.update_traces(mode='markers+lines',
                        marker=dict(size=4, # marker size
                                    line=dict(width=2)), # outline width
                        line=dict(width=1) # thinner connecting lines
        )
            
    fig.show()

    return
    

- Testing for XRV3000

In [97]:
plotly_all_eng(sub_df, "gr_tlbr", "Gantry 2", "XRV-3000", 0, "2026-01-01", 24)

- Testing for XRV4000 specifally for Post-ISM

In [98]:
plotly_all_eng(sub_df, "gr_tlbr", "Gantry 1", "XRV-4000", 0, "2026-01-01", 24)

## Test 1.4 - Ploting all energies of 1 position across all profiles in one gantry at a particular gantry angle 

In [99]:
def plotly_one_pos(df, pos, gantry, device, gantry_angle, end_date, n_months):
    '''
    Plot gradient ratio time series data using plotly module:
    - All energy at a particular gantry at a specific GA at certain position
    - Filter for XRV-3000 or XRV-4000
    - For each energy, the gradient ratio in all profiles at one position


    e.g. plotly_all_pos(sub_df, "Top-Left", "Gantry 1", "XRV-4000", 70, 0, "2026-01-01", 24)

    Input:
        df:             Pandas dataframe

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        end_date:       The end date of the data in string (e.g. "2026-01-01")

        n_months:        Number of months you want to go back from the end date in integer

    Return:
        Interactive plotly graph.
    '''

     # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                     (df["Device"] == device) & 
                     (df['ADate'] >= start_date) & 
                     (df["Gantry Angle"] == gantry_angle)
    ]

    # plot details
    subplot_titles = [prof for prof in prof_titles] # subplot titles

    # defining the subplots inside the main plot
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.05,
        vertical_spacing=0.1
    )

    plot_df = selected_df[(selected_df["Spot"]==pos)] # filtering for the specify position

    energy_list = plot_df["Energy"].unique() # all the unique energies 
    clr_no = len(energy_list) # no of energies
    colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours of the plots

    for i, prof in enumerate(profiles):

        # looping for row/col index
        total_cols = 2
        row = i // total_cols + 1
        col = i % total_cols + 1

        for j, energy in enumerate(energy_list): # looping through all energies 

            prof_rad = plot_df[(plot_df["Energy"]==energy)] # the specify profile of 1 energy

            fig.add_trace(
            go.Scatter(
                x=prof_rad['ADate'], 
                y=prof_rad[prof], 
                name=f"{energy} MeV", # labelling the lines
                legendgroup=str(energy), # grouping the same profile into 1 legend
                showlegend=(i==0), # ensure only 1 set of legend is shown
                line=dict(color=colours[j]) # defining the colour of the plots according to the colour dict
            ), 
                row=row, 
                col=col
        )   
            
    # plot settings
    fig.update_layout(
        title=dict(
            text=f"<u>{gantry} - Gradient ratio of the {pos} spot at GA{gantry_angle} for all energies on the {device}</u>",
            x=0.5
        ),
        width=1280,
        height=720,
        margin=dict(l=50, r=50, t=100, b=50),
    )

    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=4, # marker size
                                line=dict(width=2)), # outline width
                    line=dict(width=1) # thinner connecting lines
    )

    fig.show()

    return

- Testing for XRV3000

In [100]:
plotly_one_pos(sub_df, "Left", "Gantry 2", "XRV-3000", 0, "2026-01-01", 24)

- Testing for XRV4000 

In [101]:
plotly_one_pos(sub_df, "Bottom-Bottom-Centre", "Gantry 1", "XRV-4000", 0, "2026-01-01", 24)

## Test 2.1 - Plotting radar graphs of one spot with a slider to visualise the change in spot shape

- Start by plotting one radar graph of the left and right gradient of all 4 profiles

In [102]:
def radar_one_spot_test(df, gantry, device, gantry_angle, energy, pos, date):
    '''
    Plot radar graph of the left and right gradient of all 4 profiles of a single spot. 
    - One energy at a particular gantry at a specific gantry angle at certain position
    - Filter for XRV-3000 or XRV-4000

    e.g. plotly_all_pos(sub_df, "Gantry 1", "XRV-4000", 0, 70, "Centre, "2026-01-01")

    Input:
        df:             Pandas dataframe

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)
        
        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        date:       The date of the data in string (e.g. "2026-01-01")


    Return:
        Interactive plotly graph.
    '''

    selected_df = df[(df["MachineName"]==gantry) & 
                        (df["Device"] == device) &  
                        (df["Gantry Angle"] == gantry_angle) &
                        (df['ADate'] >= date) &
                        (df["Energy"] == energy) &
                        (df["Spot"] == pos)               
    ]

    grad_prof_list = ["hor_rt_gradient", "bltr_rt_gradient", "vert_lt_gradient", "tlbr_lt_gradient", # list of profile labels
                    "hor_lt_gradient", "bltr_lt_gradient", "vert_rt_gradient", "tlbr_rt_gradient"] # anti-clockwise starting from the right
    theta = grad_prof_list + [grad_prof_list[0]] # to close off the graph
    
    prof_grad = abs(selected_df[grad_prof_list].values[0]) # data
    r= np.append(prof_grad, prof_grad[0]) # to close off the graph

    fig = go.Figure()

    # plot
    fig.add_trace(
        go.Scatterpolar(
            r=r,
            theta=theta,
            mode="lines",
            line=dict(color="hsl(210, 70%, 55%)"), 
            fill="toself" # transparent fill
        )
    )

    # plot settings

    # label for date
    dates = sorted(selected_df["ADate"].unique())
    d_label = pd.to_datetime(dates).strftime("%Y-%m-%d")[-1]

    # calculation for range
    low_range = np.floor(r.min())
    upper_range = np.ceil(r.max())

    fig.update_layout(
        title=dict(
            text=f"<u>{gantry} - Spot gradients of {energy} MeV spot at GA{gantry_angle} ({device}) measured on {d_label}</u>", # title
            x=0.5
        ),
        width=800,
        height=400,
        polar=dict(
            radialaxis=dict(visible=True,
                            range=[low_range,upper_range]
            )
        )
    )

    fig.show()

    return

- Testing for XRV-3000

In [103]:
gantry = "Gantry 1"
device = "XRV-3000"
gantry_angle = 0
energy = 70
position = "Centre"
date = pd.Timestamp("2026-01-01")

radar_one_spot_test(sub_df, gantry, device, gantry_angle, energy, position, date)

- Include a slider to scroll through the date

In [104]:
def radar_one_spot(df, gantry, device, gantry_angle, energy, pos, end_date, n_months):
    '''
    Plot radar graph of the left and right gradient of all 4 profiles of a single spot. 
    - One energy at a particular gantry at a specific gantry angle at certain position
    - Filter for XRV-3000 or XRV-4000

    e.g. plotly_all_pos(sub_df, "Gantry 1", "XRV-4000", 0, 70, "Centre, "2026-01-01")

    Input:
        df:             Pandas dataframe

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)
        
        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        date:       The date of the data in string (e.g. "2026-01-01")


    Return:
        Interactive plotly graph.
    '''
    # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                        (df["Device"] == device) &  
                        (df["Gantry Angle"] == gantry_angle) &
                        (df['ADate'] >= start_date) &
                        (df["Energy"] == energy) &
                        (df["Spot"] == pos)               
    ]

    ##########
    # Figure #
    ##########

    fig = go.Figure()

    # dates for the slider
    dates = sorted(selected_df["ADate"].unique())
    trace_dates = []

    # list of profile labels
    grad_prof_list = ["hor_rt_gradient", "bltr_rt_gradient", "vert_lt_gradient", "tlbr_lt_gradient", 
                      "hor_lt_gradient", "bltr_lt_gradient", "vert_rt_gradient", "tlbr_rt_gradient"] # anti-clockwise starting from the right
    
    # calculation for range
    all_grad = abs(selected_df[grad_prof_list].values) # data

    lower_range = np.floor(all_grad.min()) # round down the lowest grad
    upper_range = np.ceil(all_grad.max()) # round up the largest grad

    for d in dates:
        sub_df= selected_df[selected_df["ADate"] == d]

        if sub_df.empty:
            continue

        # data
        prof_grad = abs(sub_df[grad_prof_list].values[0]) 
        r= np.append(prof_grad, prof_grad[0]) # to close off the graph

        theta = ["hor_rt", "bltr_rt", "vert_lt", "tlbr_lt", 
                 "hor_lt", "bltr_lt", "vert_rt", "tlbr_rt"] # radar axes label
        theta = theta + [theta[0]] # to close off the profile labels 

        # plot
        fig.add_trace(
            go.Scatterpolar(
                r=r,
                theta=theta,
                mode="lines",
                line=dict(color="hsl(210, 70%, 55%)"), 
                fill="toself" # transparent fill
            )
        )

        trace_dates.append(d)

    # starting the trace on the earliest date
    for i, trace in enumerate(fig.data):
        trace.visible = (i == 0)

    ##################
    # Graph Features #
    ##################

    # label for date
    dates = sorted(selected_df["ADate"].unique())
    d_label = pd.to_datetime(d).strftime("%Y-%m-%d")

    # info for the slider
    steps = []
    n = len(fig.data)

    # loop for the slider
    for i, d in enumerate(trace_dates):
        visible = [False] * n
        visible[i] = True

        steps.append(dict(
            method="update",
            args=[{"visible": visible}],
            label=pd.to_datetime(d).strftime("%Y-%m-%d")
        ))

    fig.update_layout(
        title=dict(
            text=f"<u>{gantry} - Profile gradients of the {energy} MeV {pos} spot at GA{gantry_angle} on the {device}</u>", # title
            x=0.5
        ),
        sliders=[dict(
            active=0,
            steps=steps
        )],
        polar=dict(
            radialaxis=dict(visible=True,
                            range=[0,upper_range]
            )
        ),
        width=900,
        height=600
    )

    fig.show()

    return

- Testing a random normal spot

In [105]:
gantry = "Gantry 2"
device = "XRV-3000"
gantry_angle = 0
energy = 70
position = "Centre"
date = pd.Timestamp("2026-01-01")
n = 48

radar_one_spot(sub_df, gantry, device, gantry_angle, energy, position, date, n)

- Testing a spot with a large grad change (e.g. G1, GA0, 70MeV, Bottom-Bottom Centre)

In [106]:
gantry = "Gantry 1"
device = "XRV-4000"
gantry_angle = 0
energy = 70
position = "Bottom-Bottom-Centre"
date = pd.Timestamp("2026-01-01")
n = 24

radar_one_spot(sub_df, gantry, device, gantry_angle, energy, position, date, n)

## Test 2.2 - Plotting radar graphs of all spots with a slider to visualise the change in spot shape

In [107]:
def radar_all_spot(df, gantry, device, gantry_angle, energy, end_date, n_months):
    '''
    Plot radar graph of the left and right gradient of all 4 profiles of a all spots. 
    - One energy at a particular gantry at a specific gantry angle across all positions
    - Filter for XRV-3000 or XRV-4000

    e.g. plotly_all_spot(sub_df, "Gantry 1", "XRV-4000", 0, 70, "2026-01-01", 12)

    Input:
        df:             Pandas dataframe

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)
        
        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        date:       The date of the data in string (e.g. "2026-01-01")


    Return:
        Interactive plotly graph.
    '''
    # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                        (df["Device"] == device) &  
                        (df["Gantry Angle"] == gantry_angle) &
                        (df['ADate'] >= start_date) &
                        (df["Energy"] == energy)              
    ]

    # list of profile headings
    grad_prof_list = ["hor_rt_gradient", "bltr_rt_gradient", "vert_lt_gradient", "tlbr_lt_gradient", 
                      "hor_lt_gradient", "bltr_lt_gradient", "vert_rt_gradient", "tlbr_rt_gradient"] # anti-clockwise starting from the right

    # calculation for range
    all_grad = abs(selected_df[grad_prof_list].values) # data
    upper_range = np.ceil(all_grad.max()) # round up the largest grad


    ##########
    # Figure #
    ##########

    fig = go.Figure()

    # graph differences for 3000 and 4000
    if device == "XRV-3000":
        spot_list = xrv3000_spot
        n_rows = 3
        sub_v = 0.05
        subtitle_gap = 0
        plot_height = 960
        sub_loop = 10

    elif device == "XRV-4000":
        spot_list = xrv4000_spot
        n_rows = 5
        sub_v = 0.04
        subtitle_gap = 0.01
        plot_height = 1200
        sub_loop = 16

    # plot details
    subplot_titles = [f"{pos}" for pos in spot_list] # subplot titles

    # defining the subplots inside the main plot
    fig = make_subplots(rows=n_rows,
                        cols=3,
                        subplot_titles=subplot_titles,
                        specs=[[{"type":"polar"}] * 3 for i in range(n_rows)],
                        horizontal_spacing=0.08,
                        vertical_spacing=sub_v
    )

    # subplot title settings
    for ann in fig.layout.annotations:
        ann.y += subtitle_gap
        ann.font.size = 12

    all_spot_traces = [] # traces for all spots

    for i, spot in enumerate(spot_list):

        # looping for row/col index
        total_cols = 3
        row = i // total_cols + 1
        col = i % total_cols + 1 

        plot_df = selected_df[(selected_df["Spot"]==spot)] # going through each spot positions

        # dates for the slider
        dates = sorted(plot_df["ADate"].unique())
        spot_traces = [] # individual trace of the current loop
        
        for d in dates:
            sub_df= plot_df[plot_df["ADate"] == d]

            if sub_df.empty:
                continue

            # data
            prof_grad = abs(sub_df[grad_prof_list].values[0]) 
            r = np.append(prof_grad, prof_grad[0]) # to close off the graph

            theta = ["hor_rt", "bltr_rt", "vert_lt", "tlbr_lt", 
                        "hor_lt", "bltr_lt", "vert_rt", "tlbr_rt"] # radar axes label
            theta = theta + [theta[0]] # to close off the profile labels 

            # plot
            fig.add_trace(
                go.Scatterpolar(
                    r=r,
                    theta=theta,
                    mode="lines",
                    line=dict(color="hsl(210, 70%, 55%)"), 
                    fill="toself", # transparent fill
                    name=str(d)
                ),
                row=row,
                col=col
            )

            spot_traces.append((d, len(fig.data) - 1)) # traces of one spot

        # traces look up
        all_spot_traces.append(spot_traces) # traces of all spots

    spot_map = {}
    for spot in all_spot_traces:
        for datetime, idx in spot:
            spot_map.setdefault(datetime, []).append(idx)

    # starting the trace on the earliest date
    for trace in fig.data: # make all traces invisible
        trace.visible = False

    for idx in spot_map[sorted(spot_map.keys())[0]]: # only showing traces of first date
        fig.data[idx].visible = True

    ###########
    # Sliders #
    ###########

    # info for the slider
    steps = [] # empty list to update the slider

    # loop for the slider
    for d in sorted(spot_map.keys()):

        visible = [False] * len(fig.data) 

        for idx in spot_map.get(d,[]):
            visible[idx] = True # showing plots when the slider is on the date

        # slider labels
        d_label = pd.to_datetime(d).strftime("%Y-%m-%d") # date label

        steps.append(dict(
            method="update",
            args=[{"visible": visible}],
            label=d_label)
        )

    top_slider = dict(
        active=0,
        x=0.5,
        y=1.15,
        xanchor="center",
        yanchor="top",
        len=0.7,
        steps=steps
    ) 

    bottom_slider = dict(
        active=0,
        x=0.5,
        y=-0.1,
        xanchor="center",
        yanchor="bottom",
        len=0.7,
        steps=steps
    ) 

    ##################
    # Graph Settings #
    ##################

    # general settings
    fig.update_layout(
        title=dict(text=f"<u>{gantry} - Profile gradients of the {energy} MeV spot at GA{gantry_angle} on the {device}</u>",
                    x=0.5,
                    y=0.99
        ), # title
        showlegend=False, # no legend
        sliders=[top_slider, bottom_slider], # slider
        width = 900,
        height = plot_height
    )

    # settings for each subplot
    for n in range(1, sub_loop):
        key = "polar" if i == 1 else f"polar{n}" # creating key to loop through each subplot

        # applying settings to each subplot
        fig.update_layout(**{
            key :dict(radialaxis=dict(visible=True,
                                    range=[0,upper_range]), # radial axes
                    angularaxis=dict(tickfont=dict(size=8)) # axes labels
                ) 
        })

    fig.show()

    return

- Testing for XRV3000

In [108]:
gantry = "Gantry 2"
device = "XRV-3000"
gantry_angle = 0
energy = 150
date = pd.Timestamp("2026-01-01")
n = 24

radar_all_spot(sub_df, gantry, device, gantry_angle, energy, date, n)

- Testing for XRV4000

In [109]:
gantry = "Gantry 1"
device = "XRV-4000"
gantry_angle = 0
energy = 70
date = pd.Timestamp("2026-01-01")
n = 24

radar_all_spot(sub_df, gantry, device, gantry_angle, energy, date, n)

## Test 2.3 - Plotting radar graphs of all spots with a slider and energy toggle

- Also manipulate the data such that the gradient order is reversed, as a higher gradient means a smaller width
- Equation used:
$$
r_2 = 2r_{\mathrm{mean}} - r
$$

In [110]:
def radar_all_spot2(df, gantry, device, gantry_angle, energy, end_date, n_months):
    '''
    Plot radar graph of the left and right gradient of all 4 profiles of all spots. 
    - One energy at a particular gantry at a specific gantry angle across all positions
    - Filter for XRV-3000 or XRV-4000

    e.g. radar_all_spot2(sub_df, "Gantry 1", "XRV-4000", 0, 70, "2026-01-01", 12)

    Input:
        df:             Pandas dataframe

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)
        
        energy:         Energy of the spot in integer (i.e. 70, 100, 150, 200, 240)

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        date:       The date of the data in string (e.g. "2026-01-01")


    Return:
        Interactive plotly graph.
    '''
    # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                        (df["Device"] == device) &  
                        (df["Gantry Angle"] == gantry_angle) &
                        (df['ADate'] >= start_date) &
                        (df["Energy"] == energy)              
    ]

    # list of profile headings
    grad_prof_list = ["hor_rt_gradient", "bltr_rt_gradient", "vert_lt_gradient", "tlbr_lt_gradient", 
                      "hor_lt_gradient", "bltr_lt_gradient", "vert_rt_gradient", "tlbr_rt_gradient"] # anti-clockwise starting from the right

    # calculation for range
    all_grad = abs(selected_df[grad_prof_list].values) # data
    upper_range = np.ceil(all_grad.max()) # round up the largest grad


    ##########
    # Figure #
    ##########

    fig = go.Figure()

    # graph differences for 3000 and 4000
    if device == "XRV-3000":
        spot_list = xrv3000_spot
        n_rows = 3
        sub_v = 0.05
        subtitle_gap = 0
        plot_height = 960
        sub_loop = 10

    elif device == "XRV-4000":
        spot_list = xrv4000_spot
        n_rows = 5
        sub_v = 0.04
        subtitle_gap = 0.01
        plot_height = 1200
        sub_loop = 16

    # plot details
    subplot_titles = [f"{pos}" for pos in spot_list] # subplot titles

    # defining the subplots inside the main plot
    fig = make_subplots(rows=n_rows,
                        cols=3,
                        subplot_titles=subplot_titles,
                        specs=[[{"type":"polar"}] * 3 for i in range(n_rows)],
                        horizontal_spacing=0.08,
                        vertical_spacing=sub_v
    )

    # subplot title settings
    for ann in fig.layout.annotations:
        ann.y += subtitle_gap
        ann.font.size = 12

    all_spot_traces = [] # traces for all spots

    for i, spot in enumerate(spot_list):

        # looping for row/col index
        total_cols = 3
        row = i // total_cols + 1
        col = i % total_cols + 1 

        plot_df = selected_df[(selected_df["Spot"]==spot)] # going through each spot positions

        # dates for the slider
        dates = sorted(plot_df["ADate"].unique())
        spot_traces = [] # individual trace of the current loop
        
        for d in dates:
            sub_df= plot_df[plot_df["ADate"] == d]

            if sub_df.empty:
                continue

            # data
            prof_grad = abs(sub_df[grad_prof_list].values[0]) 
            r = np.append(prof_grad, prof_grad[0]) # to close off the graph
            r_mean = np.mean(r)
            r2 = 2 * r_mean - r # reversing the gradient order, since a larger gradient means smaller width 

            theta = ["hor_rt", "bltr_rt", "vert_lt", "tlbr_lt", 
                        "hor_lt", "bltr_lt", "vert_rt", "tlbr_rt"] # radar axes label
            theta = theta + [theta[0]] # to close off the profile labels 

            # plot
            fig.add_trace(
                go.Scatterpolar(
                    r=r2,
                    theta=theta,
                    mode="lines",
                    line=dict(color="hsl(210, 70%, 55%)"), 
                    fill="toself", # transparent fill
                    name=str(d)
                ),
                row=row,
                col=col
            )

            spot_traces.append((d, len(fig.data) - 1)) # traces of one spot

        # traces look up
        all_spot_traces.append(spot_traces) # traces of all spots

    spot_map = {}
    for spot in all_spot_traces:
        for datetime, idx in spot:
            spot_map.setdefault(datetime, []).append(idx)

    # starting the trace on the earliest date
    for trace in fig.data: # make all traces invisible
        trace.visible = False

    for idx in spot_map[sorted(spot_map.keys())[0]]: # only showing traces of first date
        fig.data[idx].visible = True

    ###########
    # Sliders #
    ###########

    # info for the slider
    steps = [] # empty list to update the slider

    # loop for the slider
    for d in sorted(spot_map.keys()):

        visible = [False] * len(fig.data) 

        for idx in spot_map.get(d,[]):
            visible[idx] = True # showing plots when the slider is on the date

        # slider labels
        d_label = pd.to_datetime(d).strftime("%Y-%m-%d") # date label

        steps.append(dict(
            method="update",
            args=[{"visible": visible}],
            label=d_label)
        )

    top_slider = dict(
        active=0,
        x=0.5,
        y=1.15,
        xanchor="center",
        yanchor="top",
        len=0.7,
        steps=steps
    ) 

    bottom_slider = dict(
        active=0,
        x=0.5,
        y=-0.1,
        xanchor="center",
        yanchor="bottom",
        len=0.7,
        steps=steps
    ) 

    ##################
    # Graph Settings #
    ##################

    # general settings
    fig.update_layout(
        title=dict(text=f"<u>{gantry} - Profile gradients of the {energy} MeV spot at GA{gantry_angle} on the {device}</u>",
                    x=0.5,
                    y=0.99
        ), # title
        showlegend=False, # no legend
        sliders=[top_slider, bottom_slider], # slider
        width = 900,
        height = plot_height
    )

    # settings for each subplot
    for n in range(1, sub_loop):
        key = "polar" if i == 1 else f"polar{n}" # creating key to loop through each subplot

        # applying settings to each subplot
        fig.update_layout(**{
            key :dict(radialaxis=dict(visible=True,
                                    range=[0,upper_range]), # radial axes
                    angularaxis=dict(tickfont=dict(size=8)) # axes labels
                ) 
        })

    fig.show()

    return

- Testing for XRV4000

In [111]:
gantry = "Gantry 2"
device = "XRV-4000"
gantry_angle = 0
energy = 70
date = pd.Timestamp("2026-01-01")
n = 24

radar_all_spot2(sub_df, gantry, device, gantry_angle, energy, date, n)

- Testing for XRV3000

In [112]:
gantry = "Gantry 2"
device = "XRV-3000"
gantry_angle = 0
energy = 150
date = pd.Timestamp("2026-01-01")
n = 24

radar_all_spot2(sub_df, gantry, device, gantry_angle, energy, date, n)

## Test 2.4 - Plotting radar graphs with all energy

In [113]:
def radar_all_energy(df, gantry, device, gantry_angle, end_date, n_months):
    '''
    Plot radar graph of the left and right gradient of all 4 profiles of all spots. 
    - All energies at a particular gantry at a specific gantry angle across all positions
    - Filter for XRV-3000 or XRV-4000

    e.g. plotly_all_pos(sub_df, "Gantry 1", "XRV-4000", 0, "2026-01-01", 12)

    Input:
        df:             Pandas dataframe

        gantry:         Name of the gantry in string (i.e. "Gantry 1", "Gantry 2", "Gantry 3", "Gantry 4")

        device:         Name of the detector in string (i.e. "XRV-3000", "XRV-4000")

        gantry_angle:   Gantry angle in integer (i.e. 0, 90, 180, 270)

        pos:            Column name of the dataframe in string; the different spot positions (e.g. "Top-Left")

        date:       The date of the data in string (e.g. "2026-01-01")


    Return:
        Interactive plotly graph.
    '''

    ###################
    # Data Processing #
    ###################

    # only show data from last 12 months
    start_date = pd.Timestamp(end_date) - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & 
                        (df["Device"] == device) &  
                        (df["Gantry Angle"] == gantry_angle) &
                        (df['ADate'] >= start_date)           
    ]

    # energy list
    energy_list = [70, 100, 150, 200, 240]

    # list of profile headings
    grad_prof_list = ["hor_rt_gradient", "bltr_rt_gradient", "vert_lt_gradient", "tlbr_lt_gradient", 
                      "hor_lt_gradient", "bltr_lt_gradient", "vert_rt_gradient", "tlbr_rt_gradient"] # anti-clockwise starting from the right

    # calculation for range
    all_grad = abs(selected_df[grad_prof_list].values) # data
    upper_range = np.ceil(all_grad.max()) # round up the largest grad
    
    ##########
    # Figure #
    ##########

    fig = go.Figure()

    # graph differences for 3000 and 4000
    if device == "XRV-3000":
        spot_list = xrv3000_spot
        n_rows = 3
        sub_v = 0.05
        subtitle_gap = 0
        plot_height = 960
        sub_loop = 10

    elif device == "XRV-4000":
        spot_list = xrv4000_spot
        n_rows = 5
        sub_v = 0.04
        subtitle_gap = 0.01
        plot_height = 1200
        sub_loop = 16

    # plot details
    subplot_titles = [f"{pos}" for pos in spot_list] # subplot titles
    clr_no = len(energy_list) # total no. of energies
    colours = sns.color_palette("hls", clr_no).as_hex() # defining the colours for the legend of the plot

    # defining the subplots inside the main plot
    fig = make_subplots(rows=n_rows,
                        cols=3,
                        subplot_titles=subplot_titles,
                        specs=[[{"type":"polar"}] * 3 for i in range(n_rows)],
                        horizontal_spacing=0.08,
                        vertical_spacing=sub_v
    )

    # subplot title settings
    for ann in fig.layout.annotations:
        ann.y += subtitle_gap
        ann.font.size = 12

    
    # For the slider
    dates = sorted(selected_df["ADate"].unique()) # dates label
    all_spot_traces = [] # traces for all spots

    for i, spot in enumerate(spot_list):
        # looping for row/col index
        total_cols = 3
        row = i // total_cols + 1
        col = i % total_cols + 1 

        spot_df = selected_df[(selected_df["Spot"]==spot)] # going through each spot position

        # calculating the average spot profile for the r2 calculation for all energies 
        all_spot_prof = abs(spot_df[grad_prof_list].values)
        spot_prof_list = [prof for sublist in all_spot_prof for prof in sublist] # collapsing the gradient profile list into 1 
        r_mean = np.mean(spot_prof_list)


        for d in dates:
            date_df = spot_df[spot_df["ADate"] == d] # filter to a specific date
            spot_traces = [] # individual trace of the current loop

            # skip if there is no data
            if date_df.empty:
                continue

            for j, eng in enumerate(energy_list):

                eng_df = date_df[(date_df["Energy"]==eng)] # going through each energy for each position
                
                # skip if there is no data
                if eng_df.empty:
                    continue

                # data
                prof_grad = abs(eng_df[grad_prof_list].values[0]) # only 1 sub list in the list of profile gradient of an energy 
                r = np.append(prof_grad, prof_grad[0]) # to close off the graph
                r2 = 2 * r_mean - r # reversing the gradient order, since a larger gradient means smaller width 


                theta = ["hor_rt", "bltr_rt", "vert_lt", "tlbr_lt", 
                        "hor_lt", "bltr_lt", "vert_rt", "tlbr_rt"] # radar axes label
                theta = theta + [theta[0]] # to close off the profile labels 

                # plot
                fig.add_trace(
                    go.Scatterpolar(
                        r = r2,
                        theta=theta,
                        name=f"{eng} MeV", # labeling the radar's energy
                        legendgroup=str(eng), # grouping the same energy into 1 legend
                        showlegend=(i==0), # ensure only 1 set of legend is shown 
                        mode="lines",
                        line=dict(color=colours[j]), 
                        fill="toself" # transparent fill
                    ),
                    row=row,
                    col=col
                )

                spot_traces.append((d, len(fig.data) - 1)) # traces of one spot

            # traces look up
            all_spot_traces.append(spot_traces) # traces of all spots

    spot_map = {}
    for spot in all_spot_traces:
        for datetime, idx in spot:
            spot_map.setdefault(datetime, []).append(idx)

    # starting the trace on the earliest date
    for trace in fig.data: # make all traces invisible
        trace.visible = False

    for idx in spot_map[sorted(spot_map.keys())[0]]: # only showing traces of first date
        fig.data[idx].visible = True

    ###########
    # Sliders #
    ###########

    # info for the slider
    steps = [] # empty list to update the slider

    # loop for the slider
    for d in sorted(spot_map.keys()):

        visible = [False] * len(fig.data) 

        for idx in spot_map.get(d,[]):
            visible[idx] = True # showing plots when the slider is on the date

        # slider labels
        d_label = pd.to_datetime(d).strftime("%Y-%m-%d") # date label

        steps.append(dict(
            method="update",
            args=[{"visible": visible}],
            label=d_label)
        )

    top_slider = dict(
        active=0,
        x=0.5,
        y=1.15,
        xanchor="center",
        yanchor="top",
        len=0.7,
        steps=steps
    ) 

    bottom_slider = dict(
        active=0,
        x=0.5,
        y=-0.1,
        xanchor="center",
        yanchor="bottom",
        len=0.7,
        steps=steps
    ) 

    ##################
    # Graph Settings #
    ##################

    # general settings
    fig.update_layout(
        title=dict(text=f"<u>{gantry} - Profile gradients of the {energy} MeV spot at GA{gantry_angle} on the {device}</u>",
                    x=0.5,
                    y=0.99
        ), # title
        showlegend=True, # no legend
        sliders=[top_slider, bottom_slider], # slider
        width = 900,
        height = plot_height
    )

    # settings for each subplot
    for n in range(1, sub_loop):
        key = "polar" if i == 1 else f"polar{n}" # creating key to loop through each subplot

        # applying settings to each subplot
        fig.update_layout(**{
            key :dict(radialaxis=dict(visible=True,
                                    range=[0,upper_range]), # radial axes
                    angularaxis=dict(tickfont=dict(size=8)) # axes labels
                ) 
        })

    fig.show()
    
    return

In [114]:
gantry = "Gantry 2"
device = "XRV-4000"
gantry_angle = 0
date = pd.Timestamp("2026-01-01")
n = 24.

radar_all_energy(sub_df, gantry, device, gantry_angle, date, n)